# DESC Weak Lensing (WL) Task Force metrics - MAF implementation and demo


- Author: Sylvie Dagoret-Campagne
- Creation date: 2026-09-10
- Context: SCOC (Survey Cadence Optimization Committee) - DESC Task Force metrics, restricted to the DESC metrics only (3x2pt, Weak Lensing, Supernovae)
- This notebook: **WL** (Weak Lensing systematics-mitigation proxy metrics)
- Companion notebooks in this series (`06_MAF_DESC_TaskF`): `01_3x2pts_DESC_TaskForce_demo.ipynb` (3x2pt static-probes FoM, already done); a **SN** notebook is still to come.
- Simulation analyzed: `/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db`
- Same approach as notebook 01: reproduces the exact chain used by the official `rubin_sim.maf.batches.science_radar_batch` "Cosmology" / "2: WL" subgroup, so results are directly comparable to the standard show_maf pages for this OpSim run. https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/batches/science_radar_batch.py


## Notebook overview

**What the DESC WL Task Force tracks.** Unlike the 3x2pt static-probes FoM (notebook 01), which folds everything down to a single dark-energy Figure of Merit, the WL Task Force metrics available in `rubin_sim.maf` are *proxy* metrics for weak-lensing **systematics mitigation** - they do not attempt to forecast a cosmological constraint at all. The class docstring is explicit about this:

> "A proxy metric for WL systematics. Higher values indicate better systematics mitigation."

Two complementary proxies are implemented in `rubin_sim.maf.metrics.weak_lensing_systematics_metric`:

1. **`WeakLensingNvisits`** - per Healpix pixel, counts the number of good (exposure time above a minimum) visits in a chosen band combination (typically `gri` or `riz`), after applying the same Galactic-extinction and coadded-depth cuts as `ExgalM5WithCuts` (notebook 01). More visits over the usable footprint means lower shape-noise and better control of shear-measurement systematics (PSF modeling, charge-transfer inefficiency, etc. average down with more independent visits).
2. **`RIZDetectionCoaddExposureTime`** - per Healpix pixel, sums the exposure time of `riz` visits (after a dust cut and a minimum-exposure-time cut on individual visits, and requiring coverage in all `ugrizY` bands). This is a proxy for depth fluctuations in the `riz` coadd used for object *detection* by metadetection-style shear estimators (the detection scheme increasingly adopted for LSST weak lensing), independent of any explicit depth threshold.

Both metrics share the two-stage MAF structure already used for 3x2pt: a per-pixel (`parent`) metric producing a Healpix map, reduced to survey-level numbers by `summary_metrics` (`Mean`, `Median`, `Rms`, effective area). There is no FoM-emulator summary step here - these are already the final numbers used for cadence comparison.

This notebook reproduces the official `science_radar_batch` "WL" subgroup: `WeakLensingNvisits` for the full 10-year survey and per year (years 1-9) in both `gri` and `riz`, plus `RIZDetectionCoaddExposureTime` per year, all on the same reduced non-DD footprint, `nside=64`, `E(B-V) < 0.2`.


## Simulation and code references
- OpSim run analyzed: `baseline_v5.3.6_10yrs.db` (Rubin baseline v5.3.6, 10-year simulation)
- `rubin_sim.maf` source (main branch, retrieved for this notebook):
  - `rubin_sim/maf/metrics/weak_lensing_systematics_metric.py` (`WeakLensingNvisits`, `RIZDetectionCoaddExposureTime`, and `ExgalM5WithCuts`  https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/metrics/weak_lensing_systematics_metric.py used in notebook 01)
  - `rubin_sim/maf/metrics/exgal_m5.py` (`ExgalM5`, used internally by both WL metrics)
  - `rubin_sim/maf/batches/science_radar_batch.py` (official "Cosmology" batch definition, "WL" subgroup, function `science_radar_batch`)
- summary.h5 / MAF outputs for standard runs: https://s3df.slac.stanford.edu/data/rubin/sim-data/sims_featureScheduler_runs5.3/maf/
- Table of simulations: https://usdf-maf.slac.stanford.edu/
- LSST survey strategy : https://github.com/lsst-pst/survey_strategy/

## 1. Imports

In [ ]:
import os
import inspect
from os.path import splitext, basename

import numpy as np
import pandas as pd
import healpy as hp
import matplotlib.pyplot as plt

import rubin_sim
import rubin_sim.maf as maf
import rubin_sim.maf.metrics as metrics
import rubin_sim.maf.slicers as slicers
import rubin_sim.maf.maps as maps
import rubin_sim.maf.metric_bundles as mb

print("rubin_sim version:", rubin_sim.__version__)

## 2. Configuration

Same OpSim file and output-directory convention as notebook 01, with a dedicated `NB_TAG`.

In [ ]:
opsim_fname = "/Users/dagoret/DATA/OpSim/sim_baseline/baseline_v5.3.6_10yrs.db"
assert os.path.isfile(opsim_fname), f"OpSim database not found: {opsim_fname}"

run_name = splitext(basename(opsim_fname))[0]
print("run_name:", run_name)

In [ ]:
NB_TAG = "WL"
data_dir = f"data_02_{NB_TAG}"
figs_dir = f"figs_02_{NB_TAG}"
os.makedirs(data_dir, exist_ok=True)
os.makedirs(figs_dir, exist_ok=True)
print("MAF output (data) directory :", os.path.abspath(data_dir))
print("Figures output directory    :", os.path.abspath(figs_dir))

In [ ]:
resultsDb = maf.db.ResultsDb(out_dir=data_dir)

## 3. The MAF metric classes used for the WL systematics-mitigation proxies


In [ ]:
from rubin_sim.maf.metrics.weak_lensing_systematics_metric import (
    WeakLensingNvisits,
    RIZDetectionCoaddExposureTime,
)

print(inspect.getdoc(WeakLensingNvisits))
print("-" * 80)
print(inspect.getdoc(RIZDetectionCoaddExposureTime))

## 4. Year-dependent configuration (matching the official `science_radar_batch` "WL" subgroup)

Same `i`-band coadded-depth cuts as notebook 01 (used here only to define the usable footprint for the `WeakLensingNvisits` depth cut), same `nside=64`, same `E(B-V)` limit and minimum visit exposure time.

In [ ]:
bandpass = "i"
lim_ebv = 0.2
nside = 64
min_exp_time = 15
offset = 0.1

mag_cuts = {
    1: 24.75 - offset,
    2: 25.12 - offset,
    3: 25.35 - offset,
    4: 25.50 - offset,
    5: 25.62 - offset,
    6: 25.72 - offset,
    7: 25.80 - offset,
    8: 25.87 - offset,
    9: 25.94 - offset,
    10: 26.00 - offset,
}
max_year = max(mag_cuts.keys())
per_year_years = list(range(1, max_year))  # 1..9: matches the official batch's np.arange(1, 10)

pix_area = hp.nside2pixarea(nside, degrees=True)
print(f"nside={nside} -> pixel area = {pix_area:.4f} deg^2")
print("Per-year bundles computed for years:", per_year_years)
print(f"Full-survey (10 yr) bundle uses the year-{max_year} depth cut and no 'night' cut.")

## 5. Running the metric chain

For **`WeakLensingNvisits`**, we reproduce exactly the official batch structure:
- one **full-survey** bundle (all 10 years, `gri` bands, depth cut of year 10) - this is the headline number quoted for a cadence's WL performance;
- one bundle **per year** (1 to 9), separately for the `gri` and `riz` band combinations, each using that year's depth cut.

For **`RIZDetectionCoaddExposureTime`**, we compute one bundle per year (1 to 9), with `riz` as the detection bands (no explicit depth cut, by construction - see the docstring above).

In [ ]:
dustmap = maps.DustMap(nside=nside, interp=False)
slicer = slicers.HealpixSlicer(nside=nside, use_cache=False)
standard_stats = [metrics.MeanMetric(), metrics.MedianMetric(), metrics.RmsMetric()]


def band_sql(bands, night_cut=None):
    band_clause = " or ".join(f"band='{b}'" for b in bands)
    sql = f"scheduler_note not like 'DD%' and ({band_clause})"
    if night_cut is not None:
        sql += " and night < %i" % night_cut
    return sql


def run_nvisits_bundle(bands, depth_cut, night_cut, metric_name):
    m = metrics.WeakLensingNvisits(
        lsst_filter=bandpass,
        depth_cut=depth_cut,
        ebvlim=lim_ebv,
        min_exp_time=min_exp_time,
        metric_name=metric_name,
    )
    sqlconstraint = band_sql(bands, night_cut)
    bundle = mb.MetricBundle(
        m,
        slicer,
        sqlconstraint,
        maps_list=[dustmap],
        run_name=run_name,
        info_label=f"{''.join(bands)} band non-DD" + (f" year<{night_cut}" if night_cut else ""),
        summary_metrics=standard_stats,
    )
    return bundle


def run_exptime_bundle(year, night_cut, metric_name):
    m = metrics.RIZDetectionCoaddExposureTime(
        det_bands=["r", "i", "z"],
        ebvlim=lim_ebv,
        min_expTime=min_exp_time,
        metric_name=metric_name,
    )
    sqlconstraint = band_sql(["g", "r", "i"], night_cut)
    bundle = mb.MetricBundle(
        m,
        slicer,
        sqlconstraint,
        maps_list=[dustmap],
        run_name=run_name,
        info_label=f"gri band non-DD year<{night_cut}",
        summary_metrics=standard_stats,
    )
    return bundle

In [ ]:
bundle_list = []

# Full-survey gri WeakLensingNvisits (headline WL metric for this cadence)
full_survey_bundle = run_nvisits_bundle(
    bands=["g", "r", "i"],
    depth_cut=mag_cuts[max_year],
    night_cut=None,
    metric_name="WeakLensingNvisits",
)
bundle_list.append(full_survey_bundle)

gri_bundles, riz_bundles, exptime_bundles = {}, {}, {}
for year in per_year_years:
    night_cut = int(year * 365.25)
    gri_bundles[year] = run_nvisits_bundle(
        bands=["g", "r", "i"],
        depth_cut=mag_cuts[year],
        night_cut=night_cut,
        metric_name=f"WeakLensingNvisits_gri_year{year}",
    )
    bundle_list.append(gri_bundles[year])

    riz_bundles[year] = run_nvisits_bundle(
        bands=["r", "i", "z"],
        depth_cut=mag_cuts[year],
        night_cut=night_cut,
        metric_name=f"WeakLensingNvisits_riz_year{year}",
    )
    bundle_list.append(riz_bundles[year])

    exptime_bundles[year] = run_exptime_bundle(
        year=year,
        night_cut=night_cut,
        metric_name=f"gri_exposure_time_year{year}",
    )
    bundle_list.append(exptime_bundles[year])

bd = mb.make_bundles_dict_from_list(bundle_list)
bgroup = mb.MetricBundleGroup(bd, opsim_fname, out_dir=data_dir, results_db=resultsDb)
bgroup.run_all()
print("Done:", len(bundle_list), "bundles run.")

## 6. Results table

In [ ]:
rows = []
for year in per_year_years:
    sv_gri = gri_bundles[year].summary_values
    sv_riz = riz_bundles[year].summary_values
    sv_exp = exptime_bundles[year].summary_values
    rows.append(
        {
            "year": year,
            "i_depth_cut": mag_cuts[year],
            "nvisits_gri_mean": sv_gri.get("Mean"),
            "nvisits_gri_median": sv_gri.get("Median"),
            "nvisits_riz_mean": sv_riz.get("Mean"),
            "nvisits_riz_median": sv_riz.get("Median"),
            "riz_exptime_mean_s": sv_exp.get("Mean"),
            "riz_exptime_median_s": sv_exp.get("Median"),
        }
    )
results_df = pd.DataFrame(rows).set_index("year")

full_survey_row = pd.Series(
    {
        "Mean": full_survey_bundle.summary_values.get("Mean"),
        "Median": full_survey_bundle.summary_values.get("Median"),
        "Rms": full_survey_bundle.summary_values.get("Rms"),
    },
    name="WeakLensingNvisits (full 10 yr survey, gri)",
)

print("Full-survey (10 yr) headline WL metric:")
display(full_survey_row.to_frame().T)
results_df

In [ ]:
results_csv = os.path.join(data_dir, f"{run_name}_WL_results_by_year.csv")
results_df.to_csv(results_csv)
print("Saved:", results_csv)

## 7. Healpix maps and histograms

As in notebook 01, we use the Healpix slicer's own default `HealpixSkyMap` + `HealpixHistogram` plotters (`bundle.plot()`), which is exactly how MAF itself renders these per-pixel results.

In [ ]:
def save_bundle_plots(bundle, tag, figs_dir):
    made_plots = bundle.plot(savefig=False)
    saved = []
    for plot_type, fig in made_plots.items():
        if fig is None:
            continue
        base = os.path.join(figs_dir, f"{tag}_{plot_type}")
        fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
        fig.savefig(base + ".pdf", bbox_inches="tight")
        saved.append(base)
        plt.close(fig)
    return saved

In [ ]:
sample_years = [1, 5, 9]
prefix = f"{run_name}_WeakLensingNvisits_gri"
for year in sample_years:
    print(f"--- Year {year}: WeakLensingNvisits (gri) ---")
    saved = save_bundle_plots(gri_bundles[year], f"{prefix}_year{year:02d}", figs_dir)
    for s in saved:
        print("  saved:", s + ".png/.pdf")

print("--- Full survey (10 yr): WeakLensingNvisits (gri) ---")
saved = save_bundle_plots(full_survey_bundle, f"{prefix}_full10yr", figs_dir)
for s in saved:
    print("  saved:", s + ".png/.pdf")

In [ ]:
# Display the sky map + histogram for the full 10-year survey inline
_ = full_survey_bundle.plot(savefig=False)
plt.show()

In [ ]:
# Also save maps/histograms for the riz-coadd detection exposure time, year 9
prefix_exp = f"{run_name}_riz_detcoadd_exptime"
saved = save_bundle_plots(exptime_bundles[9], f"{prefix_exp}_year09", figs_dir)
for s in saved:
    print("  saved:", s + ".png/.pdf")
_ = exptime_bundles[9].plot(savefig=False)
plt.show()

## 8. Evolution of the WL proxy metrics with survey year

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(7, 10), sharex=True)

axs[0].plot(results_df.index, results_df["nvisits_gri_mean"], marker="o", color="darkgreen", label="gri")
axs[0].plot(results_df.index, results_df["nvisits_riz_mean"], marker="s", color="darkorange", label="riz")
axs[0].axhline(
    full_survey_bundle.summary_values.get("Mean"),
    color="darkgreen",
    linestyle="--",
    alpha=0.5,
    label="gri, full 10 yr",
)
axs[0].set_ylabel("Mean WeakLensingNvisits\n[visits/pixel]")
axs[0].legend()
axs[0].grid(alpha=0.3)

axs[1].plot(results_df.index, results_df["nvisits_gri_median"], marker="o", color="darkgreen", label="gri")
axs[1].plot(results_df.index, results_df["nvisits_riz_median"], marker="s", color="darkorange", label="riz")
axs[1].set_ylabel("Median WeakLensingNvisits\n[visits/pixel]")
axs[1].legend()
axs[1].grid(alpha=0.3)

axs[2].plot(results_df.index, results_df["riz_exptime_median_s"], marker="o", color="purple")
axs[2].set_ylabel("Median riz-coadd\ndetection exptime [s]")
axs[2].set_xlabel("Survey year")
axs[2].grid(alpha=0.3)

fig.suptitle(f"DESC WL Task Force proxy metrics - {run_name}")
fig.tight_layout()

base = os.path.join(figs_dir, f"{run_name}_WL_summary_vs_year")
fig.savefig(base + ".png", dpi=150, bbox_inches="tight")
fig.savefig(base + ".pdf", bbox_inches="tight")
print("Saved:", base + ".png/.pdf")
plt.show()

## 9. Caveats

- `WeakLensingNvisits` and `RIZDetectionCoaddExposureTime` are deliberately simple **proxy** metrics: more visits / more exposure time over a clean (low-extinction, sufficiently deep) footprint is *assumed* to correlate with better weak-lensing systematics control, but neither metric propagates an actual shear- or PSF-systematics budget the way a full image-simulation-based study would. They are cheap to evaluate on every OpSim run and are meant for *relative* comparison between cadences, not as an absolute systematics forecast.
- The `gri` vs `riz` band-combination choice reflects two different possible detection/shear-measurement schemes under discussion for LSST weak lensing; both are tracked side by side by the official batch, which is why this notebook reports both.
- Unlike the 3x2pt notebook, there is no FoM-emulator step here, so there is no known upstream bug to work around; if you run into a similar issue with these metrics, check `rubin_sim/maf/metrics/weak_lensing_systematics_metric.py` directly against your installed version.
- For the DESC static-probes 3x2pt FoM (which *does* combine clustering + lensing into a single dark-energy Figure of Merit), see the companion `01_3x2pts_DESC_TaskForce_demo.ipynb` notebook.


## References
- `rubin_sim.maf` documentation: https://rubin-sim.lsst.io/maf.html
- `rubin_sim` source: https://github.com/lsst/rubin_sim (`rubin_sim/maf/metrics/`, `rubin_sim/maf/batches/science_radar_batch.py`)
- Bianco, F. B. et al. 2022, "Optimization of the Observing Cadence for the Rubin Observatory LSST: A Pioneering Process of Community-Focused Experimental Design", ApJS 258, 1 - SCOC context.
